# Schema Design for Data Engineering


## Why Schema Design Matters

Bad schema = slow queries + messy data + pain

Good schema = fast queries + clean pipelines + scalable systems

👉 You don’t just query data  
👉 You design how data lives


## Bad design example

Single wide table:

`orders`:
- id
- user_name
- product_name
- price

**Problems:**
- Duplicate data
- Hard to update
- Poor performance


## What is normalization?

Split data into logical tables to:
- Reduce duplication
- Improve consistency

**Typical split:**
- `users`
- `products`
- `orders` (facts / line items)


## Prerequisites

- PostgreSQL running ([`README.md`](README.md)).
- This notebook **drops** `orders`, `products`, `users` (lesson names) if present so DDL matches this lesson.


## Create normalized tables


In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="de_course",
    user="admin",
    password="admin",
)

cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS orders CASCADE")
cur.execute("DROP TABLE IF EXISTS products CASCADE")
cur.execute("DROP TABLE IF EXISTS users CASCADE")

cur.execute(
    """
CREATE TABLE users (
    id SERIAL PRIMARY KEY,
    name TEXT
)
"""
)

cur.execute(
    """
CREATE TABLE products (
    id SERIAL PRIMARY KEY,
    name TEXT,
    price INT
)
"""
)

cur.execute(
    """
CREATE TABLE orders (
    id SERIAL PRIMARY KEY,
    user_id INT,
    product_id INT
)
"""
)

conn.commit()
print("Normalized tables created.")


## Insert data


In [ ]:
cur.execute("INSERT INTO users (name) VALUES ('Amar'), ('Neha')")
cur.execute(
    "INSERT INTO products (name, price) VALUES ('Phone', 1000), ('Laptop', 2000)"
)

cur.execute(
    """
INSERT INTO orders (user_id, product_id)
VALUES
(1,1),
(1,2),
(2,1)
"""
)

conn.commit()


## Query normalized data


In [ ]:
cur.execute(
    """
SELECT users.name, products.name, products.price
FROM orders
JOIN users ON users.id = orders.user_id
JOIN products ON products.id = orders.product_id
"""
)

print(cur.fetchall())


## Denormalization (important tradeoff)

**What is denormalization?**

Sometimes we:
- Combine tables
- Duplicate data

**Why?**
👉 Faster reads / simpler analytics queries

**Used in:**
- Data warehouses
- Wide **fact** tables / star schemas

**Tradeoff:** storage + drift vs join cost at query time.


## Indexing (performance)


In [ ]:
cur.execute("CREATE INDEX IF NOT EXISTS idx_orders_user_id ON orders(user_id)")
conn.commit()
print("Index idx_orders_user_id created (if not exists).")

# Indexes:
# - Speed up filters/joins on user_id
# - Extra cost on inserts/updates — another tradeoff


## Real-world design thinking

Ask:

1. What queries will run most?
2. What data changes frequently?
3. What needs to scale?

👉 Design for **usage**, not theory alone.


## Practice

1. Add `categories` table
2. Link `products` to categories
3. Query: **revenue per category** (use `SUM(price)` or line totals as you model it)


## Assignment

Design schema for an **e-commerce** system:

**Tables:** `users`, `products`, `orders`, `payments`

Document:
- Relationships (1:N, optional ER sketch)
- Primary / foreign keys
- Indexes you’d add first

**Bonus:** `created_at` / `updated_at` on transactional tables.


## Interview Questions

1. What is normalization?
2. When would you denormalize?
3. What is indexing?
4. How do you design a schema?


## What you just learned

- How to **structure** relational data
- **Trade-offs:** consistency vs read speed, index cost vs scan cost

👉 How senior engineers reason before writing DDL.


---

**Next:** **SQL project** — schema + seed + analytics queries (portfolio-sized).


## Optional: close connection


In [ ]:
cur.close()
conn.close()
print("Connection closed.")


## Your tasks

- [ ] Run **Create normalized tables** → **Indexing** with Postgres up (fresh kernel).
- [ ] **Practice:** `categories`, FK from `products`, aggregate revenue per category.
- [ ] **Assignment:** written or SQL DDL stub for users/products/orders/payments + keys/indexes; **bonus:** timestamps (`TIMESTAMPTZ`).
- [ ] Answer interview questions; list **one** time you’d accept duplicated denormalized columns.
